# 06 — Gold RENAMU (EAV: Unpivot)
## Transformación Silver → Gold

El RENAMU contiene **1360+ columnas-pregunta** que deben transformarse a un
modelo **Entidad-Atributo-Valor (EAV)**, convirtiendo cada columna en una fila
con los campos `pregunta` y `respuesta`.

Para evitar OutOfMemoryError en el plan de Spark, el procesamiento se hace
en **lotes (chunks)** de columnas.

**Fuente:** `data/silver/renamu/`  
**Destino:** `data/gold/stg_renamu/` (Parquet EAV)

Estructura final:
```
stg_renamu
  Ubigeo | Departamento | Provincia | Distrito |
  ccdd | ccdi | ccpp | idmunici | pregunta | respuesta
```


In [ ]:
import sys, time
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pyspark.sql import functions as F
from pyspark.sql.types import StringType

from src.paths import SILVER, GOLD, str_path, ensure_dirs
from src.spark_utils import get_spark
from core.audit.control_manager import ControlManager, ExecutionStatus

print("Imports OK")

In [ ]:
SILVER_PATH = str_path(SILVER["renamu"])
GOLD_PATH   = str_path(GOLD["stg_renamu"])

# Tamaño del lote de preguntas (cuántas columnas procesar por vez)
# Reducir a 50 si hay errores de memoria
CHUNK_SIZE = 100

# Columnas de identificación de la municipalidad (no son preguntas)
ID_COLS_CANDIDATES = [
    "ubigeo", "departamento", "provincia", "distrito",
    "ccdd", "ccdi", "ccpp", "idmunici",
    "UBIGEO", "DEPARTAMENTO", "PROVINCIA", "DISTRITO",
    "CCDD", "CCDI", "CCPP", "IDMUNICI", "ANO_RENAMU"
]

print(f"Silver: {SILVER_PATH}")
print(f"Gold  : {GOLD_PATH}")
print(f"Chunk size: {CHUNK_SIZE} preguntas/lote")
ensure_dirs()

In [ ]:
spark = get_spark(app_name="MEF_Gold_RENAMU", memory="4g")

## PARTE 1: Lectura Silver y Detección de Columnas

In [ ]:
silver_dir = SILVER["renamu"]
if not silver_dir.exists() or not any(silver_dir.rglob("*.parquet")):
    raise FileNotFoundError("Sin datos Silver RENAMU. Ejecutar: 03_silver_renamu.ipynb")

df = spark.read.parquet(SILVER_PATH)
n_municipalidades = df.count()
total_cols = len(df.columns)

print(f"Municipalidades RENAMU: {n_municipalidades:,}")
print(f"   Columnas totales: {total_cols}")

# Detectar columnas de identificación presentes en el DataFrame
id_cols = [c for c in df.columns if c.lower() in [x.lower() for x in ID_COLS_CANDIDATES]]
print(f"   Columnas ID detectadas: {id_cols}")

# Las preguntas son todas las columnas que NO son de identificación
question_cols = [c for c in df.columns if c not in id_cols]
n_chunks = (len(question_cols) + CHUNK_SIZE - 1) // CHUNK_SIZE
print(f"   Columnas-pregunta: {len(question_cols)} → {n_chunks} lotes de {CHUNK_SIZE}")

## PARTE 2: Unpivot por Lotes (Chunked EAV)

Cada lote aplica `STACK()` de Spark SQL para transformar N columnas en N filas.

In [ ]:
def unpivot_chunk(df, id_cols, chunk_cols):
    """
    Convierte un subconjunto de columnas (chunk_cols) a formato EAV
    usando STACK() de Spark SQL.
    """
    n = len(chunk_cols)
    # Castear todas a string para homogeneizar
    for c in chunk_cols:
        df = df.withColumn(c, F.col(c).cast(StringType()))

    # Construir la expresión STACK
    pairs = ", ".join([f"'{c}', `{c}`" for c in chunk_cols])
    stack_expr = f"STACK({n}, {pairs}) AS (pregunta, respuesta)"

    id_exprs = [F.col(c) for c in id_cols]
    return df.select(*id_exprs, F.expr(stack_expr)).filter(F.col("respuesta").isNotNull())


print(f"Función unpivot_chunk definida")
print(f"   Procesará {n_chunks} lote(s) de hasta {CHUNK_SIZE} preguntas")

In [ ]:
control = ControlManager(pipeline_name="gold_renamu")
execution = control.start(input_parameters={
    "n_municipalidades": n_municipalidades,
    "n_preguntas": len(question_cols),
    "chunk_size": CHUNK_SIZE
})

total_rows = 0
start_total = time.time()

try:
    for i, start_idx in enumerate(range(0, len(question_cols), CHUNK_SIZE), start=1):
        chunk = question_cols[start_idx : start_idx + CHUNK_SIZE]
        print(f"\n  Lote {i}/{n_chunks}: {len(chunk)} preguntas [idx {start_idx}:{start_idx+len(chunk)}]")

        chunk_df = unpivot_chunk(df, id_cols, chunk)

        # Primer lote: overwrite | Lotes siguientes: append
        mode = "overwrite" if i == 1 else "append"
        chunk_df.write.mode(mode).partitionBy("ANO_RENAMU").format("parquet").save(GOLD_PATH)

        n_chunk = chunk_df.count()
        total_rows += n_chunk
        print(f"    {n_chunk:,} filas EAV escritas (acum: {total_rows:,})")

    elapsed = time.time() - start_total
    control.end(
        status=ExecutionStatus.SUCCESS,
        output_summary={
            "municipalidades": n_municipalidades,
            "preguntas": len(question_cols),
            "total_filas_eav": total_rows,
            "elapsed_sec": round(elapsed, 1)
        }
    )
    print(f"\nGold RENAMU (EAV) completado en {elapsed:.1f}s")
    print(f"   {n_municipalidades:,} municipalidades × {len(question_cols):,} preguntas → {total_rows:,} filas EAV")

except Exception as e:
    control.log_error("GoldRenamuError", str(e))
    control.end(status=ExecutionStatus.FAILED, output_summary={"error": str(e)})
    raise

In [ ]:
# ── Verificación del resultado EAV ────────────────────────────────────────────
df_result = spark.read.parquet(GOLD_PATH)
print(f"Verificación Gold RENAMU (EAV):")
print(f"  Total filas  : {df_result.count():,}")
print(f"  Columnas     : {df_result.columns}")
print("\n  Muestra (Top preguntas con más respuestas):")
df_result.groupBy("pregunta").count().orderBy("count", ascending=False).show(10, truncate=False)